# Acquisition notebook

Communication with PM100A power-meter

In [154]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvisa as pv

In [155]:
from utils.pm100a import record_pm100a
from utils.naming import make_run_dir

In [156]:
import labmate
from labmate.acquisition_notebook import AcquisitionAnalysisManager

from datetime import datetime
from pathlib import Path

Set directory for data collection

In [132]:
DATA_DIR = "data/data_pm100a"
os.makedirs(DATA_DIR, exist_ok=True)

### Reflectivity and Transmitivity measurements

In [133]:
# Set names
# SAMPLE = "laseroptik-T004Q2"

# SAMPLE = "metallic_unknown"
# --- Cavity Mirrors --- #
# SAMPLE = "laseroptic_panda_hr"
# SAMPLE = "laseroptic_panda_roc_150"
# SAMPLE = "laseroptic_panda_coupler_5"
# MEAS = "reflectivity"

# --- EQ15B ---#
MEAS = "transmitivity"
SAMPLE = "eq15b_flat_HR426_1inch_face2"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

Saving data to: data/data_pm100a\transmitivity\2026-06-01_eq15b_flat_HR426_1inch_face2_001


In [134]:
ACQ_CELL = f"{MEAS}_{SAMPLE}"
print(ACQ_CELL)

transmitivity_eq15b_flat_HR426_1inch_face2


Check for PM100A

In [135]:
# Check available VISA resources
rm = pv.ResourceManager()
print(rm.list_resources())

('USB0::0x0699::0x039F::C010359::INSTR', 'USB0::0x1313::0x8079::P1007388::INSTR', 'ASRL1::INSTR', 'ASRL4::INSTR')


In [136]:
# Connect to PM100A power meter via PyVISA
# you can use the tool Power Meter Driver Switcher to switch between the two drivers, the PM100D.dll driver and the new TLPM.dll driver. 
# Pyvisa may not recognize the PM100A with the WinUSB driver, so you may need to switch to the Visa driver. 
pm100a = rm.open_resource('USB0::0x1313::0x8079::P1007388::INSTR')
print(pm100a.query('*IDN?'))

Thorlabs,PM100A,P1007388,2.5.0



In [137]:
# Record parameters
PM_SAMPLE_DELAY = 0.1 # seconds
PM_DURATION = 5 # seconds

# Wavelength
wavelength = 780 # nm

In [153]:
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

mean_mW, error_mW = record_pm100a(
        pm100a,
        duration_s=2,
        dt_s=PM_SAMPLE_DELAY
    )

print(f"Mean power = {mean_mW:.6f} ± {error_mW:.6f} mW")

aqm.save_acquisition(mean_mW=mean_mW, error_mW=error_mW, wavelength=wavelength, sample=SAMPLE)

INFO:1:2026_06_01__15_42_57__transmitivity_eq15b_flat_HR426_1inch_face2


Mean power = 22.476131 ± 0.032683 mW
